In [1]:
#!connect jupyter --kernel-name pythonkernel --kernel-spec python3
#r "nuget:ScottPlot, 5.0.*"

Installed Packages ScottPlot, 5.0.56

Loading extensions from `C:\Users\Colin\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

The `#!connect jupyter` feature is in preview. Please report any feedback or issues at https://github.com/dotnet/interactive/issues/new/choose.

Kernel added: #!pythonkernel

In [31]:
import sklearn.neural_network as nn
from sklearn.metrics import accuracy_score

# Aufgabe 1

In [3]:
import numpy as np

def sign(x):
    return np.where(x >= 0, 1, -1)

# thresholds
t1 = 2  # threshold for x1
t2 = 3  # threshold for x2

# Hidden layer weights: two perceptrons
W_hidden = np.array([[1.0, 0.0],   # h1 checks x1 > t1
                     [0.0, 1.0]])  # h2 checks x2 > t2
b_hidden = np.array([-t1, -t2])     # biases so net >=0 <=> xi >= ti


# Output perceptron: implements OR on hidden signs.
# Hidden outputs are in {-1,+1}. To implement OR:
# map hidden outputs to boolean-like variables: u = (h + 1)/2 in {0,1}
# OR = 1 if either u1 or u2 is 1.
# Algebraically: OR = sign( u1 + u2 - 0.5 ) -> map back to {-1,+1}
# But simpler: choose weights on h directly: w = [1,1], bias =  -0.5
# Compute net = h1 + h2 - 0.5; sign(net) yields +1 if at least one h==+1.
W_out = np.array([1.0, 1.0])
b_out = -0.5

def mlp_or(x):
    # x: array-like shape (2,) or (n,2)
    x = np.asarray(x)
    single = (x.ndim == 1)
    if single:
        x = x.reshape(1, -1)
    # hidden layer nets: (n,2)
    net_h = x.dot(W_hidden.T) + b_hidden
    h = sign(net_h)               # values in {-1,+1}
    # output net
    net_o = h.dot(W_out) + b_out
    y = sign(net_o)               # final output in {-1,+1}
    return y[0] if single else y


# Quick test
inputs = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [2.0, 0.0],
    [3.0, 0.0],
    [0.0, 1.0],
    [2.0, 3.0],
    [3.0, 4.0],
    [0.6, -1.0],   # x1 > t1
    [ -1.0, 0.0],  # x2 <= t2
])
for x in inputs:
    print(x, mlp_or(x))

[0. 0.] -1
[1. 0.] -1
[2. 0.] -1
[3. 0.] -1
[0. 1.] -1
[2. 3.] 1
[3. 4.] 1
[ 0.6 -1. ] -1
[-1.  0.] -1


In [13]:
public class Perceptron 
{
    private readonly double[] _weights;
    private readonly double _bias;
    private readonly Func<double, double> _activator;

    public Perceptron(double[] weights, double bias, Func<double, double> activator) {
        _weights = weights;
        _bias = bias;
        _activator = activator;
    }

    public double Process(double[] input) {
        
        if (input.Length != _weights.Length)
            throw new ArgumentException("input");

        var dotP = 0.0;
        foreach (var (i, w) in input.Zip(_weights))
            dotP += i * w;

        var h = dotP + _bias;
        return _activator(h);
    }
}

double Sign(double x) => x >= 0 ? 1 : -1;

In [19]:
var x1 = 2;
var x2 = 3;

In [ ]:
// One layer. 1 as soon as input is >= 2
new Perceptron([1.0], -x1, Sign).Process([2.0])

1

In [23]:
// One perceptron each to process x1 and x2
// Accomplice by setting the weights to 1 and 0. "Removing" One input value because of the dotP calculation
var p1 = new Perceptron([1.0, 0.0], -x1, Sign);
var p2 = new Perceptron([0.0, 1.0], -x2, Sign);

In [28]:
double[] a = [
    p2.Process([0, 1]),
    p2.Process([0, 2]),
    p2.Process([0, 3]),
    p2.Process([0, 4]),
    p2.Process([4, 0]),
];
a

[ -1, -1, 1, 1, -1 ]

In [ ]:

// if Sign was > and not >= simply adding a little bias to p3 should fix it
var p3 = new Perceptron([1, 1], 0, Sign);

double Mlp(double[] input) => p3.Process([p1.Process(input), p2.Process(input)]);

In [30]:
double[] a = [
    Mlp([1, 1]),
    Mlp([3, 1]),
    Mlp([1, 4]),
    Mlp([3, 4]),
];
a

[ -1, 1, 1, 1 ]

# Aufgabe 2

Die Dimensionen der Gewichtsmatrizen sind (Dimension vorherige Schicht) * (Dimension nächste Schicht)

 * w1 = 64 x 25
 * w2 = 32 x 64
 * w3 = 4 x 32 
  
Gösse des Bias Vektors ist einfach die Grösse der Schicht

 * b1 = 64
 * b2 = 32
 * b3 = 4

* a' = W[1] * x + b[1] (input * layer1 + bias 1)
* a  = func(a')
* b' = W[2] * a + b[2]
* b  = func(b')
* c' = W[3] * b + b[3]
* c  = func(c')

Könnte ein 4-Klassen Problem sein. (Einer der outputs muss >1 sein, die anderen 0)

# Aufgabe 3

## Aufgabe 3.2

### Spirale

|     | ReLU  | tanh  | Sigmoid |
| --- | ----- | ----- | ------- |
| EP  | 0.769 |       | > 1     |
| Los | 0.491 | 0.336 | 0.431   |


#### 1x5

|     | ReLU  | tanh  | Sigmoid |
| --- | ----- | ----- | ------- |
| EP  | 1,5   | 0.600 | > 1     |
| Los | 0.2   | 0.212 | 0.431   |

tanH geht manchmal auf 0.057 runter


#### 3x5
|     | ReLU  | tanh  | Sigmoid |
| --- | ----- | ----- | ------- |
| EP  | 0.5   | 0.300 | 1.5     |
| Los | 0.073 | 0.005 | 0.212   |

TanH braucht die wenigsten Epochen und hat das genauste Ergebnis

*die Features sin(X1) und sin(X2) waren an*

# Post mortem

Jede Person beschreibt in der ILIAS-Abgabe individuell(!) die Bearbeitung des jeweiligen Aufgabenblattes
zurückblickend mit ca. 200 bis 400 Wörtern. Gehen Sie dabei aussagekräftig und nachvollziehbar auf folgende Punkte ein: 
 (a) Zusammenfassung: Was wurde gemacht? 
 (b) Implementierungsdetails: Kurze Beschreibung besonders interessanter Aspekte der Umsetzung. 
 (c) Was war der schwierigste Teil bei der Bearbeitung? Wie haben Sie dieses Problem gelöst? 
 (d) Was haben Sie gelernt oder (besser) verstanden? 
 (e) Team: Mit wem haben Sie zusammengearbeitet? 
 (f) Link zum Repo mit der Lösung 



 * a: Für die erste Aufgabe sollte ein simples Perceptron Netz erstellt werden, welches eine ODER verknüpfung zwischen zwei zahlen darstellt. Für die zweite Aufgabe sollten sich Gedanken über die Berechnungen in einem Multi Layer Perceptron gemacht werden. Dazu wurden die Dimensionen des Perceptron und die Matrix Berechnung für jeden Layer aufgeschrieben.
 * b: Für die erste Aufgabe habe ich mir eine Perceptron als Klasse implementiert. An sonsten musste nichts 
 * c: Ich hatte erst Probleme, mir Vorzustellen, was ein MLP macht. Die Lösung für das Problem war, dass ich eine Perceptron Klasse implementiert habe. Dadurch wurden die Daten, die zusammen gehören logisch Gruppiert. Die Funktionsweise wird klare, als wen einfach nur irgend welche Matrizen multipliziert werden. Matrizen mag zwar der weg sein, wie das MLP am ende effizient berechnet wird. Das interessiert mich aber ehrlich gesagt nicht die Bohne, wenn ich verstehen will, was da eigentlich im Detail wann, wo passiert.
 * d: Nachdem ich verstanden hatte, was ein MLP mach, verstehe ich nun auch, wie die Matrizen für die Berechnung verwendet werden.
 * e: -
 * f: https://github.com/co1inco/IFM_Programieren3/tree/master/KI/8